In [64]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
import random
%matplotlib inline

In [65]:
words = open('../names.txt').read().splitlines()

In [66]:
stoi = {c:i+1 for i,c in enumerate(sorted(set(''.join(words))))}
stoi['.'] = 0
itos = {i:c for c, i in stoi.items()}

In [67]:
def build_dataset(data):
    block_size = 3
    X = []
    Y = []
    for word in data:
        start = [0] * (block_size-1)
        paddedWord = '.' + word + '.'
        subL = []
        for c in paddedWord:
            subL.append(stoi[c])
        for x, y in zip(subL, subL[1:]):
            start.append(x)
            X.append(start)
            Y.append(y)
            start = start[1:]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X,Y


In [80]:
#random.seed(42)
#random.shuffle(words)
#eightyP = int(.8 * len(words))
#ninetyP = int(.9 * len(words))
#Xtrain, Ytrain = build_dataset(words[:eightyP])
#Xdev, Ydev = build_dataset(words[eightyP:ninetyP])
#Xtest, Ytest = build_dataset(words[ninetyP:])
Xt, Yt = build_dataset(words[:5])
Xt.shape
Xt



tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [109]:
g = torch.Generator().manual_seed(42)
C = torch.randn(27, 2, generator=g)
W1 = torch.randn(6, 100, generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn(100, 27, generator=g, requires_grad=True)
b2 = torch.randn(27, generator=g)

In [ ]:
emb = torch.flatten(C[Xt], start_dim=1)
h = torch.tanh(emb @ W1 + b1)
logits = h @ W2 + b2
counts = torch.exp(logits)
probs = counts/counts.sum(dim=1, keepdim=True)
nll = torch.log(probs[torch.arange(32), Yt])
loss = -torch.mean(nll)
loss.backward()
loss

tensor(17.7758, grad_fn=<NegBackward0>)

In [154]:
F.cross_entropy(logits, Yt)

tensor(17.7758, grad_fn=<NllLossBackward0>)